# Sales Forecasting Demo - Advanced Hybrid Time Series Forecasting

This notebook demonstrates the **Sales Forecasting Demo**, the most advanced demonstration in the IntegratedML Pluggable Models project. It showcases sophisticated hybrid modeling techniques that combine Facebook Prophet's time series capabilities with LightGBM's gradient boosting for feature-rich business forecasting scenarios.

## 🎯 Demo Overview

**Complexity Level**: Advanced (Demo 3 of 3)
**Focus**: Third-party library integration with hybrid modeling
**Business Value**: Production-ready sales forecasting for inventory planning, budget allocation, and seasonal pattern analysis

### Key Features:
- **Hybrid Architecture**: Prophet (trend/seasonality) + LightGBM (feature learning)
- **Advanced Feature Engineering**: Lag features, rolling statistics, seasonal decomposition
- **Business Intelligence**: ROI analysis, inventory optimization, forecast evaluation
- **Production Ready**: IntegratedML integration, SQL deployment, monitoring

---

## 📚 Table of Contents

1. [Environment Setup & Imports](#1-environment-setup--imports)
2. [Data Generation & Exploration](#2-data-generation--exploration)
3. [Feature Engineering Pipeline](#3-feature-engineering-pipeline)
4. [Hybrid Model Training](#4-hybrid-model-training)
5. [Model Evaluation & Analytics](#5-model-evaluation--analytics)
6. [Business Intelligence Dashboard](#6-business-intelligence-dashboard)
7. [Production Deployment](#7-production-deployment)
8. [Advanced Analytics & Insights](#8-advanced-analytics--insights)
9. [Conclusion & Next Steps](#9-conclusion--next-steps)

---

## 1. Environment Setup & Imports

First, let's set up our environment and import all necessary libraries.

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime, timedelta
import json
import sys
import os

# Add project root to path
sys.path.append(os.path.abspath('../../../'))

# Sales forecasting modules
from demos.sales_forecasting.models.hybrid_forecasting_model import HybridForecastingModel
from demos.sales_forecasting.data.generate_sales_data import SalesDataGenerator
from demos.sales_forecasting.scripts.feature_engineering import FeatureEngineer
from demos.sales_forecasting.analytics.forecast_evaluator import ForecastEvaluator
from demos.sales_forecasting.analytics.business_intelligence import BusinessIntelligence

# Component models
from demos.sales_forecasting.models.components.prophet_component import ProphetComponent
from demos.sales_forecasting.models.components.lightgbm_component import LightGBMComponent
from demos.sales_forecasting.models.components.seasonal_analyzer import SeasonalAnalyzer
from demos.sales_forecasting.models.components.trend_detector import TrendDetector

# Configure visualization settings
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
warnings.filterwarnings('ignore')

print("✅ All imports successful!")
print(f"📊 Notebook started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 2. Data Generation & Exploration

Let's generate realistic sales data with complex seasonal patterns and external factors.

In [ ]:
# Initialize the sales data generator
print("🔄 Generating realistic sales data...")

generator = SalesDataGenerator(
    start_date='2022-01-01',
    end_date='2024-12-31',
    stores=['STORE001', 'STORE002', 'STORE003'],
    categories=['Electronics', 'Clothing', 'Home & Garden', 'Sports']
)

# Generate the dataset
sales_data, external_factors = generator.generate_full_dataset()

print(f"📈 Generated {len(sales_data):,} sales records")
print(f"🌡️ Generated {len(external_factors):,} external factor records")
print(f"📅 Date range: {sales_data['date'].min()} to {sales_data['date'].max()}")

In [ ]:
# Explore the generated data
print("📊 Sales Data Overview:")
print(sales_data.head())
print("\n📊 Sales Data Info:")
print(sales_data.info())
print("\n📊 Statistical Summary:")
print(sales_data.describe())

In [ ]:
# Visualize the sales data
fig, axes = plt.subplots(2, 2, figsize=(20, 12))

# Total sales over time
daily_sales = sales_data.groupby('date')['sales_amount'].sum()
axes[0, 0].plot(daily_sales.index, daily_sales.values, linewidth=1.5)
axes[0, 0].set_title('Total Daily Sales Over Time', fontsize=14, fontweight='bold')
axes[0, 0].set_ylabel('Sales Amount ($)')
axes[0, 0].grid(True, alpha=0.3)

# Sales by store
store_sales = sales_data.groupby(['date', 'store_id'])['sales_amount'].sum().unstack()
for store in store_sales.columns:
    axes[0, 1].plot(store_sales.index, store_sales[store], label=store, linewidth=1.5)
axes[0, 1].set_title('Daily Sales by Store', fontsize=14, fontweight='bold')
axes[0, 1].set_ylabel('Sales Amount ($)')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Sales by category
category_sales = sales_data.groupby('product_category')['sales_amount'].sum()
axes[1, 0].pie(category_sales.values, labels=category_sales.index, autopct='%1.1f%%')
axes[1, 0].set_title('Sales Distribution by Category', fontsize=14, fontweight='bold')

# Weekly seasonality
sales_data['day_of_week'] = pd.to_datetime(sales_data['date']).dt.day_name()
weekly_pattern = sales_data.groupby('day_of_week')['sales_amount'].mean()
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekly_pattern = weekly_pattern.reindex(day_order)
axes[1, 1].bar(weekly_pattern.index, weekly_pattern.values, color='skyblue')
axes[1, 1].set_title('Average Sales by Day of Week', fontsize=14, fontweight='bold')
axes[1, 1].set_ylabel('Average Sales Amount ($)')
axes[1, 1].tick_params(axis='x', rotation=45)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("📊 Data exploration complete!")

## 3. Feature Engineering Pipeline

Now let's apply advanced feature engineering to prepare our data for the hybrid model.

In [ ]:
# Focus on a specific store and category for detailed analysis
target_store = 'STORE001'
target_category = 'Electronics'

# Filter data for modeling
model_data = sales_data[
    (sales_data['store_id'] == target_store) & 
    (sales_data['product_category'] == target_category)
].copy()

# Convert to time series
model_data['date'] = pd.to_datetime(model_data['date'])
model_data = model_data.set_index('date').sort_index()
sales_series = model_data['sales_amount']

print(f"🎯 Focusing on {target_store} - {target_category}")
print(f"📊 Time series length: {len(sales_series)} days")
print(f"💰 Average daily sales: ${sales_series.mean():,.2f}")
print(f"📈 Sales range: ${sales_series.min():,.2f} - ${sales_series.max():,.2f}")

In [ ]:
# Initialize feature engineering pipeline
print("🔧 Starting advanced feature engineering...")

feature_engineer = FeatureEngineer(
    lag_periods=[1, 7, 14, 30],
    rolling_windows=[7, 14, 30],
    seasonal_periods=[7, 30, 365],
    include_holidays=True
)

# Merge with external factors
external_factors['date'] = pd.to_datetime(external_factors['date'])
external_factors = external_factors.set_index('date')

# Engineer features
engineered_features = feature_engineer.engineer_features(
    sales_series, 
    external_factors=external_factors
)

print(f"✅ Feature engineering complete!")
print(f"📊 Generated {len(engineered_features.columns)} features")
print(f"🔍 Feature names: {list(engineered_features.columns)}")

In [ ]:
# Visualize some key engineered features
fig, axes = plt.subplots(3, 2, figsize=(20, 15))

# Original sales vs lag features
axes[0, 0].plot(engineered_features.index, engineered_features['target'], label='Original Sales', linewidth=2)
axes[0, 0].plot(engineered_features.index, engineered_features['sales_lag_7'], label='7-day Lag', alpha=0.7)
axes[0, 0].plot(engineered_features.index, engineered_features['sales_lag_30'], label='30-day Lag', alpha=0.7)
axes[0, 0].set_title('Sales vs Lag Features', fontsize=14, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Rolling averages
axes[0, 1].plot(engineered_features.index, engineered_features['target'], label='Original Sales', linewidth=2)
axes[0, 1].plot(engineered_features.index, engineered_features['sales_ma_7'], label='7-day MA', alpha=0.8)
axes[0, 1].plot(engineered_features.index, engineered_features['sales_ma_30'], label='30-day MA', alpha=0.8)
axes[0, 1].set_title('Sales vs Moving Averages', fontsize=14, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Seasonal features
axes[1, 0].plot(engineered_features.index, engineered_features['month_sin'], label='Month (sin)', linewidth=2)
axes[1, 0].plot(engineered_features.index, engineered_features['month_cos'], label='Month (cos)', linewidth=2)
axes[1, 0].set_title('Monthly Seasonal Encoding', fontsize=14, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# External factors
if 'temperature' in engineered_features.columns:
    ax1 = axes[1, 1]
    ax2 = ax1.twinx()
    
    line1 = ax1.plot(engineered_features.index, engineered_features['target'], 'b-', label='Sales')
    line2 = ax2.plot(engineered_features.index, engineered_features['temperature'], 'r-', alpha=0.7, label='Temperature')
    
    ax1.set_ylabel('Sales Amount ($)', color='b')
    ax2.set_ylabel('Temperature (°C)', color='r')
    ax1.set_title('Sales vs Temperature', fontsize=14, fontweight='bold')
    
    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    ax1.legend(lines, labels, loc='upper left')
    ax1.grid(True, alpha=0.3)

# Growth rates
axes[2, 0].plot(engineered_features.index, engineered_features['growth_rate_1d'], label='1-day Growth', alpha=0.7)
axes[2, 0].plot(engineered_features.index, engineered_features['growth_rate_7d'], label='7-day Growth', alpha=0.7)
axes[2, 0].axhline(y=0, color='black', linestyle='--', alpha=0.5)
axes[2, 0].set_title('Growth Rates', fontsize=14, fontweight='bold')
axes[2, 0].set_ylabel('Growth Rate (%)')
axes[2, 0].legend()
axes[2, 0].grid(True, alpha=0.3)

# Feature correlation heatmap
numeric_features = engineered_features.select_dtypes(include=[np.number]).columns[:10]  # Top 10 features
corr_matrix = engineered_features[numeric_features].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, ax=axes[2, 1], 
            fmt='.2f', square=True, cbar_kws={'label': 'Correlation'})
axes[2, 1].set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("📊 Feature visualization complete!")

## 4. Hybrid Model Training

Now let's train our hybrid forecasting model combining Prophet and LightGBM.

In [ ]:
# Split data for training and testing
split_date = '2024-09-01'
train_data = engineered_features[engineered_features.index < split_date].copy()
test_data = engineered_features[engineered_features.index >= split_date].copy()

print(f"📊 Training data: {len(train_data)} samples ({train_data.index.min()} to {train_data.index.max()})")
print(f"🧪 Testing data: {len(test_data)} samples ({test_data.index.min()} to {test_data.index.max()})")

# Prepare training data
X_train = train_data.drop('target', axis=1)
y_train = train_data['target']
X_test = test_data.drop('target', axis=1)
y_test = test_data['target']

In [ ]:
# Initialize and configure the hybrid model
print("🤖 Initializing Hybrid Forecasting Model...")

model = HybridForecastingModel(
    prophet_config={
        'seasonality_mode': 'multiplicative',
        'yearly_seasonality': True,
        'weekly_seasonality': True,
        'daily_seasonality': False,
        'changepoint_prior_scale': 0.05,
        'seasonality_prior_scale': 10.0
    },
    lightgbm_config={
        'objective': 'regression',
        'metric': 'rmse',
        'boosting_type': 'gbdt',
        'num_leaves': 31,
        'learning_rate': 0.05,
        'feature_fraction': 0.9,
        'verbose': -1,
        'random_state': 42
    },
    ensemble_config={
        'prophet_weight': 0.6,
        'lightgbm_weight': 0.4,
        'use_dynamic_weighting': True
    }
)

print("✅ Model initialized successfully!")

In [ ]:
# Train the model
print("🚀 Training hybrid model... This may take a few minutes.")

import time
start_time = time.time()

# Fit the model
model.fit(X_train, y_train)

training_time = time.time() - start_time

print(f"✅ Model training completed in {training_time:.2f} seconds!")
print(f"🔧 Prophet component trained on {len(X_train)} samples")
print(f"🌳 LightGBM component trained with {model.lightgbm_model.best_iteration if hasattr(model, 'lightgbm_model') else 'N/A'} iterations")

In [ ]:
# Make predictions
print("🔮 Generating predictions...")

# Generate predictions for test set
predictions = model.predict(X_test)

# Generate confidence intervals
try:
    confidence_intervals = model.predict_with_confidence(X_test, confidence_level=0.95)
    print("✅ Confidence intervals generated successfully!")
except Exception as e:
    print(f"⚠️ Confidence intervals not available: {str(e)}")
    confidence_intervals = None

print(f"📊 Generated {len(predictions)} predictions")
print(f"🎯 Prediction range: ${predictions.min():,.2f} - ${predictions.max():,.2f}")
print(f"📈 Actual range: ${y_test.min():,.2f} - ${y_test.max():,.2f}")

In [ ]:
# Visualize predictions vs actual
fig, axes = plt.subplots(2, 2, figsize=(20, 12))

# Time series plot
axes[0, 0].plot(y_test.index, y_test.values, label='Actual', linewidth=2, color='blue')
axes[0, 0].plot(y_test.index, predictions, label='Predicted', linewidth=2, color='red', alpha=0.8)

if confidence_intervals is not None:
    axes[0, 0].fill_between(y_test.index, 
                           confidence_intervals['lower'], 
                           confidence_intervals['upper'], 
                           alpha=0.2, color='red', label='95% Confidence')

axes[0, 0].set_title('Forecast vs Actual Sales', fontsize=14, fontweight='bold')
axes[0, 0].set_ylabel('Sales Amount ($)')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Scatter plot: Predicted vs Actual
axes[0, 1].scatter(y_test, predictions, alpha=0.6, s=50)
min_val = min(y_test.min(), predictions.min())
max_val = max(y_test.max(), predictions.max())
axes[0, 1].plot([min_val, max_val], [min_val, max_val], 'r--', alpha=0.8, linewidth=2)
axes[0, 1].set_xlabel('Actual Sales ($)')
axes[0, 1].set_ylabel('Predicted Sales ($)')
axes[0, 1].set_title('Predicted vs Actual (Scatter)', fontsize=14, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# Residuals plot
residuals = y_test - predictions
axes[1, 0].plot(y_test.index, residuals, alpha=0.7, color='green')
axes[1, 0].axhline(y=0, color='black', linestyle='--', alpha=0.8)
axes[1, 0].set_title('Forecast Residuals', fontsize=14, fontweight='bold')
axes[1, 0].set_ylabel('Residual ($)')
axes[1, 0].grid(True, alpha=0.3)

# Residuals distribution
axes[1, 1].hist(residuals, bins=20, alpha=0.7, color='green', edgecolor='black')
axes[1, 1].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[1, 1].set_title('Residuals Distribution', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Residual ($)')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("📊 Prediction visualization complete!")

## 5. Model Evaluation & Analytics

Let's evaluate our model's performance using comprehensive metrics and business analytics.

In [ ]:
# Initialize the forecast evaluator
print("📊 Initializing comprehensive forecast evaluation...")

evaluator = ForecastEvaluator(
    cost_parameters={
        'stockout_cost_per_unit': 5.0,
        'holding_cost_per_unit': 0.5,
        'revenue_per_unit': 25.0,
        'ordering_cost': 100.0
    }
)

# Prepare data for evaluation
forecast_intervals = None
if confidence_intervals is not None:
    forecast_intervals = {
        '0.95': {
            'lower': confidence_intervals['lower'],
            'upper': confidence_intervals['upper']
        }
    }

# Evaluate the forecast
evaluation_results = evaluator.evaluate_forecast(
    actual=y_test,
    forecast=pd.Series(predictions, index=y_test.index),
    forecast_intervals=forecast_intervals
)

print("✅ Forecast evaluation completed!")

In [ ]:
# Display accuracy metrics
print("📊 FORECAST ACCURACY METRICS")
print("=" * 40)

accuracy = evaluation_results['accuracy_metrics']
print(f"📈 Mean Absolute Percentage Error (MAPE): {accuracy['mape']:.2f}%")
print(f"📊 Mean Absolute Error (MAE): ${accuracy['mae']:,.2f}")
print(f"📉 Root Mean Square Error (RMSE): ${accuracy['rmse']:,.2f}")
print(f"🎯 R-squared: {accuracy['r_squared']:.3f}")
print(f"⚖️ Forecast Bias: ${accuracy['bias']:,.2f}")
print(f"📍 Directional Accuracy: {accuracy['directional_accuracy']:.1f}%")
print(f"📏 Tracking Signal: {accuracy['tracking_signal']:.2f}")

# Performance grade
mape = accuracy['mape']
if mape < 5:
    grade = 'A+ (Excellent)'
elif mape < 10:
    grade = 'A (Very Good)'
elif mape < 15:
    grade = 'B (Good)'
elif mape < 20:
    grade = 'C (Acceptable)'
else:
    grade = 'D (Needs Improvement)'

print(f"\n🏆 Overall Performance Grade: {grade}")

In [ ]:
# Display business insights
print("\n💼 BUSINESS IMPACT ANALYSIS")
print("=" * 40)

business_insights = evaluation_results['business_insights']

if 'revenue_impact' in business_insights:
    revenue = business_insights['revenue_impact']
    print(f"💰 Total Revenue Impact: ${revenue['total_impact']:,.2f}")
    print(f"📦 Stockout Cost: ${revenue['stockout_cost']:,.2f}")
    print(f"🏪 Holding Cost: ${revenue['holding_cost']:,.2f}")
    print(f"⚠️ Revenue at Risk: ${revenue['revenue_at_risk']:,.2f}")

if 'inventory_optimization' in business_insights:
    inventory = business_insights['inventory_optimization']
    print(f"\n📈 Inventory Optimization Potential:")
    print(f"💵 Potential Savings: ${inventory['potential_savings']:,.2f}")
    print(f"📉 Stockout Reduction: {inventory['stockout_reduction']:.1f}%")
    print(f"📊 Safety Stock Reduction: {inventory['safety_stock_reduction']:.1f}%")

if 'forecast_reliability' in business_insights:
    reliability = business_insights['forecast_reliability']
    print(f"\n🎯 Forecast Reliability Analysis:")
    print(f"⭐ Reliability Score: {reliability['reliability_score']:.1f}/10")
    print(f"🔄 Forecast Stability: {reliability['forecast_stability']:.3f}")
    print(f"🎲 Predictive Power: {reliability['predictive_power']:.3f}")

In [ ]:
# Generate and display comprehensive evaluation dashboard
print("🎨 Generating evaluation dashboard...")

evaluator.plot_evaluation_dashboard(
    actual=y_test,
    forecast=pd.Series(predictions, index=y_test.index),
    figsize=(24, 16)
)

print("📊 Evaluation dashboard generated!")

## 6. Business Intelligence Dashboard

Let's create an executive-level business intelligence dashboard with KPIs and strategic insights.

In [ ]:
# Initialize Business Intelligence module
print("📈 Initializing Business Intelligence Analytics...")

bi = BusinessIntelligence(
    kpi_targets={
        'revenue_growth_rate': 0.15,  # 15% growth target
        'forecast_accuracy': 0.90,    # 90% accuracy target
        'inventory_turnover': 12,     # 12 times per year
        'cost_efficiency': 0.85       # 85% efficiency target
    }
)

# Generate executive dashboard
dashboard = bi.generate_executive_dashboard(
    sales_data=y_train,  # Historical sales
    forecast_data=pd.Series(predictions, index=y_test.index)
)

print("✅ Executive dashboard generated!")

In [ ]:
# Display executive summary
print("👔 EXECUTIVE DASHBOARD")
print("=" * 50)

summary = dashboard['summary']
print(f"📅 Report Date: {summary['report_date']}")
print(f"📊 Data Period: {summary['data_period']}")
print(f"🔢 Total Periods Analyzed: {summary['total_periods']}")
print(f"🔮 Forecast Periods: {summary['forecast_periods']}")

print(f"\n📈 CORE KPIS:")
kpis = dashboard['core_kpis']
print(f"💰 Total Revenue: ${kpis['total_revenue']:,.2f}")
print(f"📊 Average Daily Revenue: ${kpis['avg_daily_revenue']:,.2f}")
print(f"📈 Revenue Growth Rate: {kpis['revenue_growth_rate']:+.1f}%")
print(f"🎯 Forecast Accuracy: {kpis['forecast_accuracy']:.1f}%")
print(f"📉 Sales Volatility: {kpis['sales_volatility']:.3f}")
print(f"📊 Trend Strength: {kpis['trend_strength']:.3f}")

print(f"\n🏥 BUSINESS HEALTH:")
health = dashboard['business_health']
print(f"⭐ Overall Health Score: {health['overall_health_score']:.1f}/10")
print(f"🔒 Stability Score: {health['stability_score']:.1f}/10")
print(f"🌱 Growth Sustainability: {health['growth_sustainability']:.1f}/10")
print(f"🎯 Performance Consistency: {health['performance_consistency']:.1f}/10")

print(f"\n📋 EXECUTIVE SUMMARY:")
print(dashboard['executive_summary'])

In [ ]:
# Calculate and display ROI metrics
print("\n💹 ROI ANALYSIS")
print("=" * 30)

investment_costs = {
    'software_development': 75000,
    'data_infrastructure': 25000,
    'training_consulting': 15000,
    'maintenance_annual': 20000
}

roi_metrics = bi.calculate_roi_metrics(
    sales_data=y_train,
    forecast_data=pd.Series(predictions, index=y_test.index),
    investment_costs=investment_costs
)

print(f"💰 Total Benefits: ${roi_metrics['total_benefits']:,.2f}")
print(f"💸 Total Costs: ${roi_metrics['total_costs']:,.2f}")
print(f"💵 Net Benefit: ${roi_metrics['net_benefit']:,.2f}")
print(f"📊 ROI: {roi_metrics['roi_percentage']:.1f}%")
print(f"⏰ Payback Period: {roi_metrics['payback_period_months']:.1f} months")
print(f"📈 NPV (5 years): ${roi_metrics['npv']:,.2f}")
print(f"🎯 Benefit-Cost Ratio: {roi_metrics['benefit_cost_ratio']:.2f}:1")

In [ ]:
# Display strategic recommendations
print("\n🎯 STRATEGIC RECOMMENDATIONS")
print("=" * 40)

recommendations = dashboard['strategic_recommendations']
for i, rec in enumerate(recommendations, 1):
    print(f"{i}. {rec}")

print("\n⚠️ RISK INDICATORS")
print("=" * 25)

risks = dashboard['risk_indicators']
for risk_type, level in risks.items():
    risk_emoji = '🔴' if level == 'High' else '🟡' if level == 'Medium' else '🟢'
    print(f"{risk_emoji} {risk_type.replace('_', ' ').title()}: {level}")

print("\n🚀 OPPORTUNITIES")
print("=" * 20)

opportunities = dashboard['opportunities']
for opp in opportunities:
    impact_emoji = '🔥' if opp['potential_impact'] == 'High' else '⭐' if opp['potential_impact'] == 'Medium' else '💡'
    print(f"{impact_emoji} {opp['type']}: {opp['description']}")
    print(f"   └── Impact: {opp['potential_impact']}, Timeframe: {opp['timeframe']}")

## 7. Production Deployment

Let's demonstrate how this model can be deployed in a production environment.

In [ ]:
# Simulate production model deployment
print("🚀 PRODUCTION DEPLOYMENT SIMULATION")
print("=" * 40)

# Save the trained model
import pickle
import os

# Create models directory if it doesn't exist
model_dir = '../models/saved_models/'
os.makedirs(model_dir, exist_ok=True)

# Save the model
model_path = os.path.join(model_dir, 'hybrid_forecasting_model.pkl')
with open(model_path, 'wb') as f:
    pickle.dump(model, f)

print(f"💾 Model saved to: {model_path}")

# Save feature engineering pipeline
feature_path = os.path.join(model_dir, 'feature_engineer.pkl')
with open(feature_path, 'wb') as f:
    pickle.dump(feature_engineer, f)

print(f"🔧 Feature engineer saved to: {feature_path}")

# Model metadata
model_metadata = {
    'model_name': 'HybridForecastingModel',
    'version': '1.0',
    'training_date': datetime.now().isoformat(),
    'training_samples': len(X_train),
    'test_samples': len(X_test),
    'features': list(X_train.columns),
    'performance_metrics': {
        'mape': float(accuracy['mape']),
        'mae': float(accuracy['mae']),
        'rmse': float(accuracy['rmse']),
        'r_squared': float(accuracy['r_squared'])
    },
    'business_impact': {
        'roi_percentage': float(roi_metrics['roi_percentage']),
        'payback_months': float(roi_metrics['payback_period_months'])
    }
}

metadata_path = os.path.join(model_dir, 'model_metadata.json')
with open(metadata_path, 'w') as f:
    json.dump(model_metadata, f, indent=2)

print(f"📄 Model metadata saved to: {metadata_path}")
print("✅ Model deployment package ready!")

In [ ]:
# Simulate real-time prediction API
def predict_sales_api(date, temperature=20.0, humidity=50.0, promotion_active=False, promotion_discount=0.0):
    """
    Simulate a production API for real-time sales forecasting.
    
    Args:
        date: Target prediction date
        temperature: Temperature forecast
        humidity: Humidity forecast  
        promotion_active: Whether promotion is active
        promotion_discount: Promotion discount rate
        
    Returns:
        dict: Prediction results with confidence intervals
    """
    try:
        # Create input data (simplified - would need full feature engineering in production)
        input_date = pd.to_datetime(date)
        
        # Basic features
        features = {
            'year': input_date.year,
            'month': input_date.month,
            'day': input_date.day,
            'dayofweek': input_date.dayofweek,
            'quarter': input_date.quarter,
            'is_weekend': int(input_date.dayofweek >= 5),
            'temperature': temperature,
            'humidity': humidity,
            'promotion_active': int(promotion_active),
            'promotion_discount': promotion_discount
        }
        
        # Add seasonal features
        features.update({
            'month_sin': np.sin(2 * np.pi * input_date.month / 12),
            'month_cos': np.cos(2 * np.pi * input_date.month / 12),
            'day_sin': np.sin(2 * np.pi * input_date.day / 31),
            'day_cos': np.cos(2 * np.pi * input_date.day / 31)
        })
        
        # Convert to DataFrame (would need proper feature engineering in production)
        input_df = pd.DataFrame([features])
        
        # Add missing features with default values
        for col in X_train.columns:
            if col not in input_df.columns:
                input_df[col] = 0.0  # Default value
        
        # Reorder columns to match training data
        input_df = input_df[X_train.columns]
        
        # Make prediction
        prediction = model.predict(input_df)[0]
        
        # Estimate confidence intervals (simplified)
        residual_std = np.std(y_test - predictions)
        lower_bound = prediction - 1.96 * residual_std
        upper_bound = prediction + 1.96 * residual_std
        
        return {
            'status': 'success',
            'prediction_date': date,
            'predicted_sales': round(prediction, 2),
            'confidence_lower': round(lower_bound, 2),
            'confidence_upper': round(upper_bound, 2),
            'model_version': '1.0',
            'timestamp': datetime.now().isoformat()
        }
        
    except Exception as e:
        return {
            'status': 'error',
            'error_message': str(e),
            'timestamp': datetime.now().isoformat()
        }

# Test the API
print("🧪 Testing Production API...")

# Test case 1: Normal day
result1 = predict_sales_api('2024-12-15', temperature=15.0, humidity=60.0)
print(f"Test 1 - Normal day: ${result1['predicted_sales']:,.2f}")

# Test case 2: Hot day with promotion
result2 = predict_sales_api('2024-12-15', temperature=30.0, humidity=40.0, 
                           promotion_active=True, promotion_discount=0.2)
print(f"Test 2 - Hot day + promotion: ${result2['predicted_sales']:,.2f}")

# Test case 3: Weekend
result3 = predict_sales_api('2024-12-14', temperature=22.0, humidity=55.0)  # Saturday
print(f"Test 3 - Weekend: ${result3['predicted_sales']:,.2f}")

print("✅ API testing completed!")

## 8. Advanced Analytics & Insights

Let's dive deeper into component analysis and advanced forecasting insights.

In [ ]:
# Analyze individual model components
print("🔍 COMPONENT ANALYSIS")
print("=" * 30)

# If the model has component predictions available
if hasattr(model, 'get_component_predictions'):
    try:
        component_preds = model.get_component_predictions(X_test)
        
        fig, axes = plt.subplots(2, 2, figsize=(20, 12))
        
        # Prophet component
        if 'prophet' in component_preds:
            axes[0, 0].plot(y_test.index, y_test.values, label='Actual', linewidth=2)
            axes[0, 0].plot(y_test.index, component_preds['prophet'], label='Prophet', linewidth=2, alpha=0.8)
            axes[0, 0].set_title('Prophet Component vs Actual', fontsize=14, fontweight='bold')
            axes[0, 0].legend()
            axes[0, 0].grid(True, alpha=0.3)
        
        # LightGBM component
        if 'lightgbm' in component_preds:
            axes[0, 1].plot(y_test.index, y_test.values, label='Actual', linewidth=2)
            axes[0, 1].plot(y_test.index, component_preds['lightgbm'], label='LightGBM', linewidth=2, alpha=0.8)
            axes[0, 1].set_title('LightGBM Component vs Actual', fontsize=14, fontweight='bold')
            axes[0, 1].legend()
            axes[0, 1].grid(True, alpha=0.3)
        
        # Component comparison
        if 'prophet' in component_preds and 'lightgbm' in component_preds:
            axes[1, 0].plot(y_test.index, component_preds['prophet'], label='Prophet', linewidth=2)
            axes[1, 0].plot(y_test.index, component_preds['lightgbm'], label='LightGBM', linewidth=2)
            axes[1, 0].plot(y_test.index, predictions, label='Ensemble', linewidth=2, alpha=0.8)
            axes[1, 0].set_title('Component Comparison', fontsize=14, fontweight='bold')
            axes[1, 0].legend()
            axes[1, 0].grid(True, alpha=0.3)
        
        # Component errors
        if 'prophet' in component_preds and 'lightgbm' in component_preds:
            prophet_error = np.abs(y_test.values - component_preds['prophet'])
            lightgbm_error = np.abs(y_test.values - component_preds['lightgbm'])
            ensemble_error = np.abs(y_test.values - predictions)
            
            x_pos = np.arange(3)
            errors = [np.mean(prophet_error), np.mean(lightgbm_error), np.mean(ensemble_error)]
            labels = ['Prophet', 'LightGBM', 'Ensemble']
            
            axes[1, 1].bar(x_pos, errors, color=['blue', 'green', 'red'], alpha=0.7)
            axes[1, 1].set_xticks(x_pos)
            axes[1, 1].set_xticklabels(labels)
            axes[1, 1].set_title('Mean Absolute Error Comparison', fontsize=14, fontweight='bold')
            axes[1, 1].set_ylabel('MAE ($)')
            axes[1, 1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        print("📊 Component analysis visualization complete!")
        
    except Exception as e:
        print(f"⚠️ Component analysis not available: {str(e)}")
else:
    print("ℹ️ Component analysis not available for this model implementation")

In [ ]:
# Feature importance analysis
print("🎯 FEATURE IMPORTANCE ANALYSIS")
print("=" * 35)

# Get feature importance if available
if hasattr(model, 'get_feature_importance'):
    try:
        feature_importance = model.get_feature_importance()
        
        # Display top features
        top_features = feature_importance.head(15)
        print("🏆 Top 15 Most Important Features:")
        for i, (feature, importance) in enumerate(top_features.items(), 1):
            print(f"{i:2d}. {feature:<25} {importance:8.4f}")
        
        # Visualize feature importance
        fig, axes = plt.subplots(1, 2, figsize=(20, 8))
        
        # Top 20 features bar plot
        top_20 = feature_importance.head(20)
        axes[0].barh(range(len(top_20)), top_20.values[::-1], color='skyblue')
        axes[0].set_yticks(range(len(top_20)))
        axes[0].set_yticklabels(top_20.index[::-1])
        axes[0].set_xlabel('Importance Score')
        axes[0].set_title('Top 20 Feature Importance', fontsize=14, fontweight='bold')
        axes[0].grid(True, alpha=0.3)
        
        # Feature categories
        lag_importance = feature_importance[feature_importance.index.str.contains('lag')].sum()
        ma_importance = feature_importance[feature_importance.index.str.contains('ma')].sum()
        seasonal_importance = feature_importance[feature_importance.index.str.contains('sin|cos|month|day')].sum()
        external_importance = feature_importance[feature_importance.index.str.contains('temp|humidity|promo')].sum()
        other_importance = feature_importance.sum() - (lag_importance + ma_importance + seasonal_importance + external_importance)
        
        categories = ['Lag Features', 'Moving Averages', 'Seasonal', 'External Factors', 'Other']
        category_importance = [lag_importance, ma_importance, seasonal_importance, external_importance, other_importance]
        
        axes[1].pie(category_importance, labels=categories, autopct='%1.1f%%', startangle=90)
        axes[1].set_title('Feature Importance by Category', fontsize=14, fontweight='bold')
        
        plt.tight_layout()
        plt.show()
        
        print("📊 Feature importance analysis complete!")
        
    except Exception as e:
        print(f"⚠️ Feature importance analysis not available: {str(e)}")
else:
    print("ℹ️ Feature importance analysis not available for this model implementation")

In [ ]:
# Generate business report
print("📄 GENERATING COMPREHENSIVE BUSINESS REPORT")
print("=" * 50)

# Generate the report
business_report = evaluator.generate_business_report()

print(business_report)

# Save the report
report_path = '../models/reports/'
os.makedirs(report_path, exist_ok=True)

report_file = os.path.join(report_path, f'sales_forecast_report_{datetime.now().strftime("%Y%m%d_%H%M%S")}.txt')
with open(report_file, 'w') as f:
    f.write(business_report)

print(f"📄 Business report saved to: {report_file}")

## 9. Conclusion & Next Steps

This comprehensive demonstration showcases the advanced capabilities of the Sales Forecasting system, representing the pinnacle of sophistication in the IntegratedML Pluggable Models project.

In [ ]:
# Final summary and conclusions
print("🎉 SALES FORECASTING DEMO COMPLETE!")
print("=" * 45)

print(f"✅ Successfully demonstrated:")
print(f"   🔧 Advanced hybrid modeling (Prophet + LightGBM)")
print(f"   📊 Comprehensive feature engineering pipeline")
print(f"   🎯 Business-oriented forecast evaluation")
print(f"   💼 Executive-level business intelligence")
print(f"   🚀 Production deployment readiness")
print(f"   📈 ROI analysis and business impact assessment")

print(f"\n📊 Model Performance Summary:")
print(f"   • MAPE: {accuracy['mape']:.2f}% ({grade})")
print(f"   • Business Impact: ${business_insights['revenue_impact']['total_impact']:,.2f}")
print(f"   • ROI: {roi_metrics['roi_percentage']:.1f}%")
print(f"   • Payback Period: {roi_metrics['payback_period_months']:.1f} months")

print(f"\n🚀 Next Steps for Production:")
print(f"   1. Deploy model using IntegratedML SQL integration")
print(f"   2. Implement real-time data pipelines")
print(f"   3. Set up automated model monitoring and retraining")
print(f"   4. Create business dashboards and alerting")
print(f"   5. Scale to multiple stores and product categories")

print(f"\n💡 Key Learnings:")
print(f"   • Hybrid models outperform individual components")
print(f"   • Feature engineering is critical for time series success")
print(f"   • Business metrics are as important as statistical metrics")
print(f"   • Production readiness requires comprehensive evaluation")

print(f"\n🎯 This demo represents the most advanced forecasting capabilities")
print(f"   in the IntegratedML Pluggable Models project, showcasing")
print(f"   production-ready hybrid modeling for business forecasting!")

print(f"\n📝 Notebook completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")